# Tutorial: O Filtro de Kalman Passo a Passo
Este notebook foi estruturado para publicação no GitHub Pages (ou renderização via Jupyter/JupyterLab). 
Aqui, traduzimos a metodologia matemática do **Filtro de Kalman** em código Python funcional, dividindo-o nos 5 passos clássicos da literatura.

O Filtro de Kalman atua em um ciclo perpétuo de **Previsão** (o que o modelo físico diz) e **Atualização/Correção** (o que o sensor diz).\n

## Exemplo 1: Filtro de Kalman Escalar (1D)
Neste cenário, estamos rastreando a posição de um objeto em uma linha reta.
Os dados iniciais no instante $k=0$ são:
* **Posição Inicial ($x_0$):** 10.0 metros
* **Incerteza Inicial ($P_0$):** 2.0
* **Ruído do Processo ($Q$):** 0.1
* **Ruído do Sensor/Medição ($R$):** 1.0

No instante $k=1$, recebemos uma medição do sensor: $z_1 = 10.5$ metros.\n

In [ ]:
# Parâmetros Iniciais (Instante k=0)
x = 10.0  # Estado inicial estimado (posição)
P = 2.0   # Covariância/Incerteza inicial
F = 1.0   # Matriz de transição de estado (modelo físico)
H = 1.0   # Matriz de observação (sensor)
Q = 0.1   # Ruído do processo (imperfeição do modelo)
R = 1.0   # Ruído da medição (imperfeição do sensor)

# Nova leitura do sensor no instante k=1
z = 10.5

print("--- FASE 1: PREVISÃO (Time Update) ---")
# Passo 1: Previsão do Estado
x_pred = F * x
print(f"Passo 1: Posição Prevista = {x_pred:.2f} m")

# Passo 2: Previsão da Covariância do Erro
P_pred = F * P * F + Q
print(f"Passo 2: Incerteza Prevista = {P_pred:.2f}\n")

print("--- FASE 2: ATUALIZAÇÃO / CORREÇÃO (Measurement Update) ---")
# Passo 3: Cálculo do Ganho de Kalman (K)
K = (P_pred * H) / (H * P_pred * H + R)
print(f"Passo 3: Ganho de Kalman (K) = {K:.3f}")

# Passo 4: Atualização do Estado
x = x_pred + K * (z - H * x_pred)
print(f"Passo 4: Posição Atualizada Final = {x:.4f} m")

# Passo 5: Atualização da Covariância do Erro
P = (1 - K * H) * P_pred
print(f"Passo 5: Nova Incerteza (P) = {P:.3f}")\n

## Exemplo 2: Filtro de Kalman Matricial (2D)
Aqui, utilizamos vetores e matrizes para rastrear 4 variáveis de estado simultaneamente: **Posição em X, Posição em Y, Velocidade em X e Velocidade em Y**. (Modelo de Velocidade Constante).

Os dados iniciais no instante $k=0$ são:
* **Estado Inicial:** Posição $(0,0)$ e Velocidade $(1,2)$ m/s
* A matriz de observação $H$ indica que o nosso sensor só consegue ler a **posição**, mas o filtro também será capaz de corrigir a velocidade indiretamente.

No instante $k=1$, recebemos uma medição de posição do sensor: $z_1 = (1.2, 2.1)$ metros.\n

In [ ]:
import numpy as np

# Parâmetros Iniciais (Instante k=0)
# Vetor de Estado [pos_x, pos_y, vel_x, vel_y]^T
x = np.array([[0.0], 
              [0.0], 
              [1.0], 
              [2.0]])

# Covariância Inicial (Incerteza)
P = np.eye(4)

# Matriz de Transição de Estado (Física: pos_atual = pos_anterior + vel * dt) (dt = 1)
F = np.array([[1, 0, 1, 0],
              [0, 1, 0, 1],
              [0, 0, 1, 0],
              [0, 0, 0, 1]])

# Matriz de Observação (O sensor apenas lê pos_x e pos_y)
H = np.array([[1, 0, 0, 0],
              [0, 1, 0, 0]])

# Matriz de Ruído do Processo (Q) e da Medição (R)
Q = 0.1 * np.eye(4)
R = np.array([[0.5, 0.0],
              [0.0, 0.5]])

# Nova leitura do sensor (pos_x, pos_y) no instante k=1
z = np.array([[1.2], 
              [2.1]])

print("--- FASE 1: PREVISÃO ---")
# Passo 1: Previsão do Estado
x_pred = F @ x
print("Passo 1: Estado Previsto (x_pred):\n", x_pred)

# Passo 2: Previsão da Covariância
P_pred = F @ P @ F.T + Q
print("\nPasso 2: Covariância Prevista (P_pred):\n", P_pred)

print("\n--- FASE 2: ATUALIZAÇÃO / CORREÇÃO ---")
# Passo 3: Ganho de Kalman (K)
# S é a inovação ou incerteza combinada
S = H @ P_pred @ H.T + R
K = P_pred @ H.T @ np.linalg.inv(S)
print("Passo 3: Ganho de Kalman (K):\n", np.round(K, 3))

# Passo 4: Atualização do Estado
x = x_pred + K @ (z - H @ x_pred)
print("\nPasso 4: Estado Atualizado (x):\n", np.round(x, 3))

# Passo 5: Atualização da Covariância
I = np.eye(4)
P = (I - K @ H) @ P_pred
print("\nPasso 5: Covariância Atualizada (P):\n", np.round(P, 3))\n